# Zhu algebras

Zhu's associative algebra of a vertex algebra $V$ graded by integer and half-integer conformal weights (untwisted; Zhu, Kac–Wang) is

$$A(V) = V_{\mathbb Z} / J, \qquad a * b = \sum_{j \ge 0} \binom{\Delta_a}{j}\, a_{(j-1)} b .$$

* $V_{\mathbb Z}$ is the span of the elements of integer weight; elements of half-integer weight map to zero.
* $J$ is spanned by $a *_{-2} b = \sum_j \binom{\Delta_a}{j} a_{(j-2)} b$ for $a, b$ of integer weight, and by $\sum_j \binom{\Delta_a - 1/2}{j} a_{(j-1)} b$ for $a, b$ of half-integer weight.

This is the notation of De Sole–Kac, *Finite vs affine W-algebras*, Japan. J. Math. 1 (2006), Section 2.2 ($\mathrm{Zhu}_\Gamma V$ with $\Gamma = \mathbb Z$). Irreducible positive-energy $V$-modules correspond to irreducible $A(V)$-modules, their top levels.

What `zhu_algebra.py` implements:

* **Universal vertex algebras**, `V.zhu_algebra()`. The algebra is generated by the images $[g]$ of the generators of integer weight, with the relations
  $$[a][b] - (-1)^{|a||b|}[b][a] = \sum_{j \ge 0} \binom{\Delta_a - 1}{j} [a_{(j)} b].$$
  Its basis is the ordered monomials in the $[g]$; products are computed by rewriting. The Zhu map $v \mapsto [v]$ is `A(v)`, also `A.retract(v)` (Sage's name for the projection to a quotient). Methods: `relations`, `check_overlaps` (the diamond lemma), `as_g_algebra`, `is_known_exact`.
* **When this is $A(V)$ itself.** De Sole–Kac (Section 3.3) prove $A(V) \cong U(\mathrm{Zhu}\,R)$, with this basis, when all generators have positive integer weight; then `is_known_exact()` is `True`. Otherwise (half-integer generators, generators of weight 0) the algebra is only known to map onto $A(V)$, and its name says so.
* **Lattice vertex algebras**, `V.zhu_algebra()`: a sum of matrix algebras over the top levels of the modules for a positive definite lattice, zero otherwise (section 8).
* **Tensor products**, `V.zhu_algebra()`: the tensor product of the factors' Zhu algebras (section 9); with quotient factors `V.zhu_algebra_up_to(W)`.
* **Quotients** $Q = V/I$, `Q.zhu_algebra_up_to(W)`. $A(V/I) = A(V)/[I]$ (Frenkel–Zhu). The images of the generators of $I$ need not generate $[I]$, so this quotients by the images of the whole weight spaces $I_w$, $w \le W$. The result maps onto $A(Q)$, shrinks as $W$ grows, and equals $A(Q)$ for $W$ large enough, which the code cannot detect. Two-sided Gröbner bases come from Singular's Plural.

Zhu algebras of a subalgebra come from its presentation (`S.presentation().zhu_algebra_up_to(W)`); higher Zhu algebras (`V.zhu_algebra(level=n)`) are in section 11, and modules over vertex algebras in `10_modules.ipynb`.

**Prerequisites:** `01_getting_started.ipynb`, `06_subalgebras_ideals_quotients.ipynb`.

**Contents**

1. Virasoro: $A(V) = \mathbb C[x]$ and the Zhu map
2. Affine $\mathfrak{sl}_2$: $U(\mathfrak{sl}_2)$ and the Casimir
3. $W_3$ and the diamond lemma
4. Odd generators: exterior and Clifford algebras
5. Half-integer weights: N=1, N=2, $\beta\gamma$
6. Quotients: minimal models and $L_k(\mathfrak g)$
7. Images of the generators of an ideal are not enough: the 3-state Potts model
8. Lattice vertex algebras
9. Tensor products
10. The H-twisted (Ramond) Zhu algebra
11. Higher Zhu algebras
12. `TestSuite`

In [1]:
import vertex_algebras as va
from vertex_algebras import *
import os, sys, time
from vertex_algebras.universal_vertex_algebra import UniversalVertexAlgebra
from vertex_algebras.normally_ordered_expressions import NormallyOrderedExpressions
from vertex_algebras.affine_vertex_algebra import UniversalAffineVertexAlgebra
K.<c> = Frac(QQ['c'])
Kk.<k> = Frac(QQ['k'])
x = polygen(QQ, 'x')


def zhu_product(a, b):
    "a * b = Σ_j C(Δ_a, j) a_(j-1) b"
    h = ZZ(QQ(a.degree()))
    return sum(binomial(h, j) * a.nproduct(b, j - 1) for j in range(h + 1))


def star_minus_2(a, b):
    "a *_{-2} b = Σ_j C(Δ_a, j) a_(j-2) b, an element of J"
    h = ZZ(QQ(a.degree()))
    return sum(binomial(h, j) * a.nproduct(b, j - 2) for j in range(h + 1))


def as_polynomial(f):
    "an element of a Zhu algebra with one generator, as a polynomial in x"
    return sum(cf * x^len(m) for m, cf in f.monomial_coefficients().items())


def singular_vectors(V, w, conditions):
    "the vectors of weight w killed by x_(n) for (x, ns) in conditions, n in ns"
    S = V.weight_space(w).basis()
    rows = []
    for g, ns in conditions:
        for n in ns:
            ims = [g.nproduct(s, n).monomial_coefficients() for s in S]
            keys = sorted(set(kk for d in ims for kk in d))
            rows += [[d.get(kk, 0) for d in ims] for kk in keys]
    if not rows:
        return S
    M = matrix(V.base_ring(), rows)
    return [sum(cf * s for cf, s in zip(r, S)) for r in M.right_kernel_matrix().rows()]

## 1. Virasoro: $A(V) = \mathbb C[x]$ and the Zhu map

The universal Virasoro vertex algebra has one generator of weight 2, so $A(V) = \mathbb C[x]$ with $x = [L]$ (Frenkel–Zhu). This is exact (`is_known_exact()`).

In [2]:
E.<L> = NormallyOrderedExpressions(K)
Vir = UniversalVertexAlgebra(K, E, {('L', 'L'): [L.derivative(), 2 * L, 0, c / 2]}, conformal_vector=L)
A = Vir.zhu_algebra()
A, A.gens(), A.is_known_exact()

(Zhu algebra of Universal vertex algebra on generators ['L'] over Fraction Field of Univariate Polynomial Ring in c over Rational Field,
 ([L],),
 True)

The Zhu map $v \mapsto [v]$ is computed recursively on the basis monomials of $V$, by the formula of De Sole–Kac, Section 3.3, for a leading letter $\partial^k a$:

$$[{:}(\partial^k a) B{:}] = k!\binom{-\Delta_a}{k}\Big([a][B] - (\Delta_a + k)\sum_j \frac{1}{k+j+1}\binom{\Delta_a - 1}{j}[a_{(j)} B]\Big).$$

`A(v)` computes $[v]$; `A.retract(v)` is the same map. It is a conversion, not a coercion: the product of $V$ is the normally ordered product, not Zhu's product. For example $[\partial L] = -2[L]$ and $[{:}LL{:}] = [L]^2 + 2[L]$:

In [3]:
L = Vir.gen('L')
for v in [Vir.one(), L, L.derivative(), L.derivative(2), L.no(L), L.no(L.no(L))]:
    print(v, '  ->  ', A(v))

1   ->   1
L   ->   [L]
dL   ->   -2*[L]
d^2L   ->   6*[L]
:L L:   ->   2*[L] + [L]^2
:L L L:   ->   8*[L] + 6*[L]^2 + [L]^3


Zhu's identities hold on any elements: $[(\partial + \Delta)a] = 0$, $[a *_{-2} b] = 0$, $[a * b] = [a][b]$, and the commutator formula. (The test suite `tests/smoke_zhu.sage` checks them on random elements, and the Zhu map against Zhu's product formula.)

In [4]:
a, b = L.no(L.derivative()), L.no(L) + 3 * L.derivative(2)
(A(a.derivative() + a.degree() * a),
 A(star_minus_2(a, b)),
 A(zhu_product(a, b)) == A(a) * A(b))

(0, 0, True)

## 2. Affine $\mathfrak{sl}_2$: $U(\mathfrak{sl}_2)$ and the Casimir

For the universal affine vertex algebra, $A(V) = U(\mathfrak g)$ (Frenkel–Zhu). `relations()` lists the supercommutators of the generators.

In [5]:
Aff = UniversalAffineVertexAlgebra(Kk, ['A', 1], level=k, names=['E', 'H', 'F'], conformal_vector='sugawara')
AA = Aff.zhu_algebra()
AA, AA.relations(), AA.is_known_exact()

(Zhu algebra of Universal affine vertex algebra of type ['A', 1] at level k over Fraction Field of Univariate Polynomial Ring in k over Rational Field,
 {('H', 'E'): 2*[E], ('F', 'E'): -[H], ('F', 'H'): 2*[F]},
 True)

The Sugawara vector $T = \frac{1}{2(k+2)}\big({:}EF{:} + {:}FE{:} + \frac12 {:}HH{:}\big)$ maps to the Casimir element divided by $2(k+2)$, which is central:

In [6]:
E_, H_, F_ = AA.gens()
zT = AA(Aff.conformal_vector())
zT, zT == (E_ * F_ + F_ * E_ + H_ * H_ / 2) / (2 * (k + 2)), all(zT * g == g * zT for g in AA.gens())

((1/(k+2))*[E]*[F] - (1/2/(k+2))*[H] + (1/4/(k+2))*[H]^2, True, True)

Products are computed by rewriting with the relations. The same algebra as Singular's G-algebra (used for the Gröbner bases of quotients):

In [7]:
print(F_ * E_ * E_)
AA.as_g_algebra()

-2*[E] + [E]^2*[F] - 2*[E]*[H]


Noncommutative Multivariate Polynomial Ring in x0, x1, x2 over Fraction Field of Univariate Polynomial Ring in k over Rational Field, nc-relations: {x1*x0: x0*x1 + 2*x0, x2*x0: x0*x2 - x1, x2*x1: x1*x2 + 2*x2}

## 3. $W_3$ and the diamond lemma

The ordered monomials are a basis of the rewriting algebra iff all overlaps resolve (Bergman's diamond lemma). `check_overlaps()` checks them: a failure means that a Jacobi identity of $V$ fails, or that $A(V)$ has further relations.

For $W_3$, $A(V)$ is commutative, $\mathbb C[x, y]$.

In [8]:
E3.<L3, W3> = NormallyOrderedExpressions(K)
Lam = L3 * L3 - 3/10 * L3.derivative(2)


def w3(beta):
    return UniversalVertexAlgebra(K, E3, {
        ('L3', 'L3'): [L3.derivative(), 2 * L3, 0, c / 2], ('L3', 'W3'): [W3.derivative(), 3 * W3],
        ('W3', 'W3'): [beta * Lam.derivative() + L3.derivative(3) / 15,
                       2 * beta * Lam + 3/10 * L3.derivative(2), L3.derivative(), 2 * L3, 0, c / 3]},
        conformal_vector=L3)


VW = w3(16 / (22 + 5 * c))
AW = VW.zhu_algebra()
AW, AW.relations(), AW.check_overlaps(), AW.is_known_exact()

(Zhu algebra of Universal vertex algebra on generators ['L3', 'W3'] over Fraction Field of Univariate Polynomial Ring in c over Rational Field,
 {},
 [],
 True)

A limitation: with only two even generators there are no overlaps, so the check cannot see anything. $W_3$ with a wrong $\beta$ (the Jacobi identity fails) still passes; the vertex algebra's own `TestSuite` (`_test_borcherds`) or `jacobi_identity_failures()` catches it.

In [9]:
bad = w3(1)
bad.zhu_algebra().check_overlaps(), len(bad.jacobi_identity_failures()) > 0

([], True)

With three generators there are overlaps. $\mathfrak{sl}_2$ currents with $[H, E] = 3E$ (not a Lie algebra) fail one:

In [10]:
broken = UniversalVertexAlgebra(QQ, ['E', 'H', 'F'], {
    ('H', 'E'): [{(('E', 0),): 3}], ('H', 'F'): [{(('F', 0),): -2}], ('E', 'F'): ['H', 1], ('H', 'H'): [0, 2]},
    degrees={'E': 1, 'H': 1, 'F': 1})
broken.zhu_algebra().check_overlaps()

[(('F', 'H', 'E'), [H])]

## 4. Odd generators: exterior and Clifford algebras

Odd generators of integer weight give odd elements of $A(V)$, which appear at most once in a basis monomial. Symplectic fermions ($\chi^\pm$ of weight 1, $\chi^+\chi^- \sim (z-w)^{-2}$) give the exterior algebra on two generators:

In [11]:
SF = UniversalVertexAlgebra(QQ, ['chp', 'chm'], {('chp', 'chm'): [0, 1]}, parities={'chp': 1, 'chm': 1},
                            degrees={'chp': 1, 'chm': 1})
ASF = SF.zhu_algebra()
p_, m_ = ASF.gens()
ASF, p_ * m_ + m_ * p_, p_ * p_, ASF.is_known_exact()

(Zhu algebra of Universal vertex algebra on generators ['chp', 'chm'] over Rational Field,
 0,
 0,
 True)

The $bc$ system with weights $(1, 0)$ gives the Clifford algebra $[b][c] + [c][b] = 1$. The generator $c$ has weight 0, outside De Sole–Kac's theorem, so the algebra is only claimed to map onto $A(V)$:

In [12]:
X.<b, cc> = NormallyOrderedExpressions(QQ, parities={'b': 1, 'cc': 1})
BC = UniversalVertexAlgebra(QQ, X, {('b', 'cc'): [1]}, conformal_vector=-b * cc.derivative())
ABC = BC.zhu_algebra()
bz, cz = ABC.gens()
ABC, bz * cz + cz * bz, bz * bz, ABC.is_known_exact()

(Algebra mapping onto the Zhu algebra of Universal vertex algebra on generators ['b', 'cc'] over Rational Field,
 1,
 0,
 False)

## 5. Half-integer weights: N=1, N=2, $\beta\gamma$

Generators of half-integer weight have zero image; elements of $J$ of the second kind give the recursion

$$[{:}xB{:}] = -\sum_{j \ge 1}\binom{\Delta_x - 1/2}{j}[x_{(j-1)}B] \qquad (\Delta_x \in \tfrac12 + \mathbb Z).$$

The algebra is generated by the images of the integer-weight generators. Whether it has further relations in general is open (De Sole–Kac, remark at the end of Section 3.3: it need not be PBW generated), so `is_known_exact()` is `False`.

N=1 (Neveu–Schwarz): one generator $[L]$, $[G] = 0$, $[{:}G\,\partial G{:}] = 4[L]$.

In [13]:
N.<Ln, G> = NormallyOrderedExpressions(K, parities={'G': 1})
N1 = UniversalVertexAlgebra(K, N, {('Ln', 'Ln'): [Ln.derivative(), 2 * Ln, 0, c / 2],
                                   ('Ln', 'G'): [G.derivative(), 3/2 * G],
                                   ('G', 'G'): [2 * Ln, 0, 2 * c / 3]}, conformal_vector=Ln)
A1 = N1.zhu_algebra()
Gv = N1.gen('G')
print(A1)
A1.gens(), A1.is_known_exact(), A1(Gv), A1(Gv.no(Gv.derivative()))

Algebra mapping onto the Zhu algebra of Universal vertex algebra on generators ['Ln', 'G'] over Fraction Field of Univariate Polynomial Ring in c over Rational Field


(([Ln],), False, 0, 4*[Ln])

N=2: the images of $L$ and $J$, commuting:

In [14]:
N2E.<L2, J2, Gp, Gm> = NormallyOrderedExpressions(K, parities={'Gp': 1, 'Gm': 1})
N2 = UniversalVertexAlgebra(K, N2E, {
    ('L2', 'L2'): [L2.derivative(), 2 * L2, 0, c / 2], ('L2', 'J2'): [J2.derivative(), J2],
    ('L2', 'Gp'): [Gp.derivative(), 3/2 * Gp], ('L2', 'Gm'): [Gm.derivative(), 3/2 * Gm],
    ('J2', 'J2'): [0, c / 3], ('J2', 'Gp'): [Gp], ('J2', 'Gm'): [-Gm],
    ('Gp', 'Gm'): [L2 + J2.derivative() / 2, J2, c / 3]}, conformal_vector=L2)
A2 = N2.zhu_algebra()
Gp_, Gm_ = N2.gen('Gp'), N2.gen('Gm')
A2.gens(), A2.relations(), A2(Gp_.no(Gm_)), A2(Gp_.derivative().no(Gm_))

(([L2], [J2]), {}, -[L2] + 1/2*[J2], [L2] - 1/2*[J2])

For N=1 and N=2 this is in fact $A(V)$. The modes form a Lie superalgebra, so Verma modules exist for every highest weight: $(h)$ for N=1, $(h, q)$ for N=2. Each gives a one-dimensional top level, on which $[L], [J]$ act by $h, q$. A polynomial vanishing at every point is zero, so $\mathbb C[x] \to A(V)$ (resp. $\mathbb C[x, y] \to A(V)$) is injective. The code cannot know this, which is why `is_known_exact()` stays `False`.

$\beta\gamma$ with weights $(\frac12, \frac12)$: no generators of integer weight, $A = \mathbb C$, and ${:}\beta\gamma{:} = \beta_{(-1)}\gamma$ is itself in $J$:

In [15]:
BG = UniversalVertexAlgebra(QQ, ['be', 'ga'], {('be', 'ga'): [1]}, degrees={'be': 1/2, 'ga': 1/2})
ABG = BG.zhu_algebra()
be, ga = BG.gens()
ABG.ngens(), ABG(be.no(ga)), ABG(BG.one())

(0, 0, 1)

## 6. Quotients: minimal models and $L_k(\mathfrak g)$

For $Q = V/I$, `Q.zhu_algebra_up_to(W)` is $A(V)$ modulo the images of the weight spaces $I_w$, $w \le W$. Below $W = \mathrm{wt}(v)$ there are no relations (dimension $\infty$). In the examples the dimension is the number (or, for $L_k(\mathfrak g)$, the sum of squares of the dimensions) of the top levels of the irreducible modules.

### Lee–Yang, $c = -22/5$

The singular vector $\Lambda = {:}LL{:} - \frac{3}{10}\partial^2 L$ maps to $x(x + \frac15)$: the conformal weights $0, -\frac15$ of the two irreducible modules.

In [16]:
EQ.<Lq> = NormallyOrderedExpressions(QQ)


def virasoro_at(cv):
    return UniversalVertexAlgebra(QQ, EQ, {('Lq', 'Lq'): [Lq.derivative(), 2 * Lq, 0, cv / 2]},
                                  degrees={'Lq': 2})


VLY = virasoro_at(-22/5)
Lg = VLY.gen('Lq')
v = Lg.no(Lg) - 3/10 * Lg.derivative(2)
ALY = VLY.zhu_algebra()
f = ALY(v)
f, as_polynomial(f).roots()

(1/5*[Lq] + [Lq]^2, [(0, 1), (-1/5, 1)])

In [17]:
QLY = VLY.quotient([v])
print([(W, QLY.zhu_algebra_up_to(W).dimension()) for W in range(3, 9)])
Z = QLY.zhu_algebra_up_to(8)
print(Z)
xz = Z.gen(0)
list(Z.basis()), xz * xz, Z.leading_monomials()

[(3, +Infinity), (4, 2), (5, 2), (6, 2), (7, 2), (8, 2)]
Quotient of Zhu algebra of Universal vertex algebra on generators ['Lq'] over Rational Field by the images of the ideal of weight <= 8: an algebra mapping onto the Zhu algebra of Quotient of Universal vertex algebra on generators ['Lq'] over Rational Field by the ideal generated by [:Lq Lq: - 3/10*d^2Lq]


([1, [Lq]], -1/5*[Lq], [(0, 0)])

### Ising, $c = \frac12$

The singular vector of weight 6 (computed as the kernel of $L_1, L_2$) maps to a cubic with roots $0, \frac1{16}, \frac12$.

In [18]:
VI = virasoro_at(1/2)
LI = VI.gen('Lq')
[vI] = singular_vectors(VI, 6, [(LI, (2, 3))])
AI = VI.zhu_algebra()
fI = AI(vI)
QI = VI.quotient([vI])
as_polynomial(fI).roots(), [(W, QI.zhu_algebra_up_to(W).dimension()) for W in range(5, 11)]

([(1/2, 1), (1/16, 1), (0, 1)],
 [(5, +Infinity), (6, 3), (7, 3), (8, 3), (9, 3), (10, 3)])

### Tricritical Ising, N=1 at $c = \frac{7}{10}$

A quotient of an algebra with half-integer generators. The first singular vector of the vacuum module (killed by $L_1, L_2, G_{1/2}, G_{3/2}$) has weight 4 and maps to $x(x - \frac1{10})$: the weights $0, \frac1{10}$ of the two irreducible Neveu–Schwarz modules.

In [19]:
N7E.<Lt, Gt> = NormallyOrderedExpressions(QQ, parities={'Gt': 1})
N7 = UniversalVertexAlgebra(QQ, N7E, {('Lt', 'Lt'): [Lt.derivative(), 2 * Lt, 0, 7/20],   # c/2, c = 7/10
                                      ('Lt', 'Gt'): [Gt.derivative(), 3/2 * Gt],
                                      ('Gt', 'Gt'): [2 * Lt, 0, 7/15]}, conformal_vector=Lt)
L7, G7 = N7.gen('Lt'), N7.gen('Gt')
cond = [(L7, range(2, 7)), (G7, range(1, 7))]
print([(w, len(singular_vectors(N7, w, cond))) for w in (2, 5/2, 3, 7/2, 4)])
[v7] = singular_vectors(N7, 4, cond)
A7 = N7.zhu_algebra()
f7 = A7(v7)
Q7 = N7.quotient([v7])
as_polynomial(f7).roots(), [(W, Q7.zhu_algebra_up_to(W).dimension()) for W in range(3, 9)]

[(2, 0), (5/2, 0), (3, 0), (7/2, 0), (4, 1)]


([(1/10, 1), (0, 1)], [(3, +Infinity), (4, 2), (5, 2), (6, 2), (7, 2), (8, 2)])

### $L_k(\mathfrak g)$ for integrable levels

$L_k(\mathfrak g) = V^k(\mathfrak g)/\langle e_\theta(-1)^{k+1}\mathbf 1\rangle$, and $A(L_k(\mathfrak g)) = \bigoplus_\lambda \mathrm{End}(V_\lambda)$ over the integrable highest weights of level $k$ (Frenkel–Zhu). So its dimension is $\sum_\lambda (\dim V_\lambda)^2$:

| | integrable top levels | $\sum (\dim V_\lambda)^2$ |
|---|---|---|
| $L_1(\mathfrak{sl}_2)$ | $1, 2$ | 5 |
| $L_2(\mathfrak{sl}_2)$ | $1, 2, 3$ | 14 |
| $L_3(\mathfrak{sl}_2)$ | $1, 2, 3, 4$ | 30 |
| $L_1(\mathfrak{sl}_3)$ | $1, 3, \bar 3$ | 19 |
| $L_1(B_2)$ | $1, 5, 4$ | 42 |

In [20]:
for ct, lev, theta in [(['A', 1], 1, 'alpha[1]'), (['A', 1], 2, 'alpha[1]'), (['A', 1], 3, 'alpha[1]'),
                       (['A', 2], 1, 'alpha[1] + alpha[2]'), (['B', 2], 1, 'alpha[1] + 2*alpha[2]')]:
    V = UniversalAffineVertexAlgebra(QQ, ct, level=lev, conformal_vector='sugawara')
    e = V.gen(theta)
    v = e
    for _ in range(lev):
        v = e.no(v)
    Q = V.quotient([v])
    print(ct, 'level', lev, [(W, Q.zhu_algebra_up_to(W).dimension()) for W in (lev, lev + 1, lev + 2)])

['A', 1] level 1 [(1, +Infinity), (2, 5), (3, 5)]
['A', 1] level 2 [(2, +Infinity), (3, 14), (4, 14)]


['A', 1] level 3 [(3, +Infinity), (4, 30), (5, 30)]
['A', 2] level 1 [(1, +Infinity), (2, 19), (3, 19)]


['B', 2] level 1 [(1, +Infinity), (2, 42), (3, 42)]


Nested quotients work too: here $Q_1 = L_1(\mathfrak{sl}_2)$ and $Q_2 = Q_1/\langle{:}FF{:}\rangle$, where ${:}FF{:}$ is already zero in $Q_1$.

In [21]:
V = UniversalAffineVertexAlgebra(QQ, ['A', 1], level=1, names=['E', 'H', 'F'], conformal_vector='sugawara')
E1, F1 = V.gen('E'), V.gen('F')
Q1 = V.quotient([E1.no(E1)])
Q2 = Q1.quotient([Q1(F1.no(F1))])
Q2.zhu_algebra_up_to(3).dimension()

5

## 7. Images of the generators of an ideal are not enough: the 3-state Potts model

For generators of weight 1 (affine algebras) the image $[v]$ of a singular vector generates $[I]$ as a two-sided ideal (Frenkel–Zhu), and taking only `A.quotient([A(v)])` gives the same dimensions as above. That argument needs all generators of $V$ to have weight 1. A generator $a$ of weight $\Delta \ge 2$ has modes $a_{(j)}$, $0 \le j < \Delta - 1$, whose images on elements of $I$ are not determined by $[v]$.

$W_3$ at $c = \frac45$ (the 3-state Potts model) shows it. The first singular vector of the vacuum has weight 6:

* The quotient by its image alone is infinite dimensional.
* The images of $I_7$ cut it down to dimension 7, and those of $I_8$ to 6. That is the number of irreducible modules: weights $0, \frac25, \frac23, \frac23, \frac1{15}, \frac1{15}$.

So `zhu_algebra_up_to` takes whole weight spaces.

In [22]:
XP = NormallyOrderedExpressions(QQ, ['T', 'W'])
T, W = XP.gens()
cP = 4/5
betaP = 16 / (22 + 5 * cP)
LmP = T * T - 3/10 * T.derivative(2)
Potts = UniversalVertexAlgebra(QQ, XP, {('T', 'T'): [T.derivative(), 2 * T, 0, cP / 2],
                                        ('T', 'W'): [W.derivative(), 3 * W],
                                        ('W', 'W'): [betaP * LmP.derivative() + T.derivative(3) / 15,
                                                     2 * betaP * LmP + 3/10 * T.derivative(2),
                                                     T.derivative(), 2 * T, 0, cP / 3]},
                            degrees={'T': 2, 'W': 3})
T, W = Potts.gens()
cond = [(T, (2, 3)), (W, (3, 4))]
print([(w, len(singular_vectors(Potts, w, cond))) for w in range(2, 7)])
[vP] = singular_vectors(Potts, 6, cond)
AP = Potts.zhu_algebra()
print('image of v:', AP(vP))
print('quotient by the image of v alone:', AP.quotient([AP(vP)]).dimension())
QP = Potts.quotient([vP])
[(Wt, QP.zhu_algebra_up_to(Wt).dimension()) for Wt in range(6, 11)]

[(2, 0), (3, 0), (4, 0), (5, 0), (6, 1)]
image of v: 14/475*[T] - 9/19*[T]^2 + [T]^3 - 117/95*[W]^2
quotient by the image of v alone: +Infinity


[(6, +Infinity), (7, 7), (8, 6), (9, 6), (10, 6)]

The eigenvalues of $[T]$ on the 6-dimensional algebra (multiplication from the left) are the conformal weights of the six modules:

In [23]:
ZP = QP.zhu_algebra_up_to(8)
B = list(ZP.basis())
tz = ZP.gen(0)
M = matrix(QQ, [[(tz * bb).monomial_coefficients().get(m, 0) for m in ZP.basis().keys()] for bb in B])
M.charpoly().factor()

(x - 2/5) * x * (x - 2/3)^2 * (x - 1/15)^2

## 8. Lattice vertex algebras

The conformal weights of a lattice vertex algebra $V_L$ are $\Delta(\mu) = \tfrac12(\mu,\mu) + \beta\cdot\mu$, whatever conformal vector or conformal weights gave them.

* **$L$ positive definite.** Every $V_L$-module is a direct sum of the $V_{\lambda+L}$, $\lambda \in L^\circ/L$ (Dong; Dong–Li–Mason), whatever the conformal vector. So $A(V_L) \cong \bigoplus_\lambda \mathrm{End}(U_\lambda)$, where the top level $U_\lambda$ of $V_{\lambda+L}$ is spanned by the $e^\mu$, $\mu \in \lambda + L$, of minimal $\Delta(\mu)$ (`top_levels()`). The basis is the matrix units $|\nu\rangle\langle\mu|$, mapping $e^\mu$ to $e^\nu$. The Zhu map sends $v$ to its zero modes on the top levels:
  $$o({:}\partial^{n_1}J_{i_1}\cdots\partial^{n_r}J_{i_r}\, e^q{:})\, e^\mu = \varepsilon(q, \mu - \lambda) \prod_j (-1)^{n_j} n_j!\, (x_{i_j}, \mu)\; e^{\mu+q}$$
  if $\mu + q$ is in the same top level, and $0$ otherwise.
* **$L$ not positive definite.** $A(V_L) = 0$. Take $\alpha \in L$ with $(\alpha,\alpha) = -2k < 0$ (doubled if $(\alpha, \alpha)$ is odd). Then $e^\alpha_{(n)} e^{-\alpha} = 0$ for $n > -2k-1$ and $e^\alpha_{(-2k-1)} e^{-\alpha} = \varepsilon\mathbf 1$. Zhu's Lemma 2.1.2 says $\mathrm{Res}_z\, Y(a,z)b\,(1+z)^{\Delta_a}/z^{2k+1} \in O(V)$, and for $a = e^\alpha$, $b = e^{-\alpha}$ this element is $\varepsilon\mathbf 1$.

In [24]:
from vertex_algebras.lattice_vertex_algebra import LatticeVertexAlgebra


def lattice_voa(B, alpha=None):
    "V_L with T = T_sug + ∂J_alpha (alpha in the coordinates of L)"
    B = matrix(ZZ, B)
    Lat = LatticeVertexAlgebra(QQ, B)
    J = Lat.heisenberg_currents()
    n = B.nrows()
    Bi = B.inverse()
    T = sum(Bi[i, j] * J[i].no(J[j]) for i in range(n) for j in range(n)) / 2
    if alpha is not None:
        T += sum(QQ(a) * J[i].derivative() for i, a in enumerate(alpha))
    return Lat.with_conformal_vector(T)


VL1 = lattice_voa([[2]])          # the root lattice of sl2: L_1(sl2)
ZL1 = VL1.zhu_algebra()
print(ZL1)
print(ZL1.dimension(), ZL1.top_levels())

Zhu algebra of Lattice vertex algebra of rank 1 with Gram matrix [2] over Rational Field
5 [(0, [(0,)]), (1/4, [(-1/2,), (1/2,)])]


The dimension 5 is that of $A(L_1(\mathfrak{sl}_2))$ in section 6. On the second top level, the images of $e^{\pm\alpha}$ and $J$ are the 2-dimensional representation of $\mathfrak{sl}_2$, and $[T]$ acts on each top level by its conformal weight. `matrices()` gives the blocks (rows are targets). Zhu's product formula holds, and $(\partial + L_0)a$ maps to zero:

In [25]:
el, fl, hl = ZL1(VL1.e((1,))), ZL1(VL1.e((-1,))), ZL1(VL1.J(0))
print(el, '|', fl, '|', hl)
print(el * fl - fl * el == hl)
a_, b_ = VL1.e((1,)), VL1.J(0).no(VL1.e((-1,)))
print(ZL1(zhu_product(a_, b_)) == ZL1(a_) * ZL1(b_), ZL1(a_.derivative() + a_) == 0)
ZL1(VL1.conformal_vector()).matrices()

|1/2><-1/2| | |-1/2><1/2| | -|-1/2><-1/2| + |1/2><1/2|
True
True True


[
     [1/4   0]
[0], [  0 1/4]
]

More lattices. $A_2$ gives the dimension 19 of $A(L_1(\mathfrak{sl}_3))$ in section 6; $D_4$ has three 8-dimensional top levels of weight $\frac12$; $E_8$ is holomorphic. A background charge changes the top levels, and so the Zhu algebra: with $T = T_{\mathrm{sug}} + \frac14\partial J$ the second module of $A_1$ has a one-dimensional top level.

In [26]:
for name, B in [('A2', CartanMatrix(['A', 2])), ('D4', CartanMatrix(['D', 4])), ('E8', CartanMatrix(['E', 8]))]:
    Zl = lattice_voa(B).zhu_algebra()
    print(name, Zl.dimension(), [(h, len(mus)) for h, mus in Zl.top_levels()])
Zb = lattice_voa([[2]], alpha=[1/4]).zhu_algebra()
print('A1 with T_sug + ∂J/4:', Zb.dimension(), Zb.top_levels())

A2 19 [(0, 1), (1/3, 3), (1/3, 3)]
D4 193 [(0, 1), (1/2, 8), (1/2, 8), (1/2, 8)]
E8 1 [(0, 1)]
A1 with T_sug + ∂J/4: 2 [(0, [(0,)]), (0, [(1/2,)])]


Odd lattices. $V_{\mathbb Z}$ with weights 1 and 0 is the $bc$ system of section 4 ($b = e^{1}$, $c = e^{-1}$); its Zhu algebra is the Clifford algebra $\mathrm{End}(\mathbb C^{1|1})$, a super algebra. $V_{\sqrt3\,\mathbb Z}$ is the N=2 algebra at $c = 1$, with three modules of weights $0, \frac16, \frac16$.

In [27]:
VZ = LatticeVertexAlgebra(QQ, matrix([[1]]), degrees={'J0': 1, 'e^(1)': 1, 'e^(-1)': 0})
Zbc = VZ.zhu_algebra()
bz, cz = Zbc(VZ.e((1,))), Zbc(VZ.e((-1,)))
print(Zbc.top_levels(), bz * cz + cz * bz == Zbc.one(), bz * bz == 0, cz * cz == 0)
print(Zbc.category())
lattice_voa([[3]]).zhu_algebra().top_levels()

[(0, [(-1,), (0,)])] True True True
Category of finite dimensional super algebras with basis over Rational Field


[(0, [(0,)]), (1/6, [(-1/3,)]), (1/6, [(1/3,)])]

A form that is not positive definite gives the zero algebra. For $I_{1,1} = \mathrm{diag}(1, -1)$, take $\alpha = (0, 2)$: $(\alpha, \alpha) = -4$, so $k = 2$, and the element of $O(V)$ from the proof is $\pm\mathbf 1$:

In [28]:
VI = lattice_voa([[1, 0], [0, -1]])
Z0 = VI.zhu_algebra()
print(Z0)
print(Z0.dimension(), Z0.one() == 0, Z0(VI.e((1, 0))) == 0)
a_, b_ = VI.e((0, 2)), VI.e((0, -2))
wa, kk = a_.degree(), 2
sum(binomial(wa, i) * a_.nproduct(b_, i - 2*kk - 1) for i in range(2*kk + 2))

Zhu algebra of Lattice vertex algebra of rank 2 with Gram matrix [[1, 0], [0, -1]] over Rational Field (zero: the form is not positive definite)
0 True True


1

## 9. Tensor products

$A(V_1 \otimes V_2) \cong A(V_1) \otimes A(V_2)$ (Frenkel–Zhu), with $[a \otimes b] = [a] \otimes [b]$. With odd elements it is the super tensor product, whose sign is the Koszul sign of $V_1 \otimes V_2$. Each factor's Zhu algebra is taken for the grading the tensor product gives that factor, so a factor needs no conformal vector of its own. The lattice algebra of $\mathfrak{sl}_2$ times the Virasoro algebra at $c = -\frac{22}{5}$:

In [29]:
EV.<Lv> = NormallyOrderedExpressions(QQ)
VirLY = UniversalVertexAlgebra(QQ, EV, {('Lv', 'Lv'): [Lv.derivative(), 2*Lv, 0, -11/5]}, conformal_vector=Lv)
Lv = VirLY.gen('Lv')
VT = VL1.tensor(VirLY, conformal_vector='factors')
ZT = VT.zhu_algebra()
print(ZT)
ZT(VT.pure_tensor(VL1.e((1,)), Lv.no(Lv)))

Zhu algebra of Tensor product of vertex algebras [Lattice vertex algebra of rank 1 with Gram matrix [2] over Rational Field, Universal vertex algebra on generators ['Lv'] over Rational Field]


2*|1/2><-1/2| # [Lv] + |1/2><-1/2| # [Lv]^2

With a quotient factor, `zhu_algebra_up_to(W)` takes the factor's `zhu_algebra_up_to(W)`: here $(\mathbb C \oplus M_2) \otimes \mathbb C^2$, of dimension 10, for $L_1(\mathfrak{sl}_2) \otimes$ Lee–Yang. Two copies of the lattice $bc$ system give $\mathrm{End}(\mathbb C^{1|1})^{\otimes 2}$, where the images of $b$ from the two factors anticommute:

In [30]:
LYq = VirLY.quotient([Lv.no(Lv) - 3/10*Lv.derivative(2)])
ZQ = VL1.tensor(LYq, degrees='factors').zhu_algebra_up_to(6)
print(ZQ.dimension(), [B.dimension() for B in ZQ.factors()], ZQ.is_known_exact())
VB2 = VZ.tensor(VZ, degrees='factors')
ZB2 = VB2.zhu_algebra()
b0, b1 = ZB2(VB2.embed(0, VZ.e((1,)))), ZB2(VB2.embed(1, VZ.e((1,))))
print(ZB2.dimension(), b0 * b1 + b1 * b0 == 0, b0 * b1)

10 [5, 2] False
16 True |0><-1| # |0><-1|


## 10. The H-twisted (Ramond) Zhu algebra

`V.zhu_algebra(twist='H')` (also `twist='ramond'`) is De Sole–Kac's H-twisted Zhu algebra $A_H(V)$ (*Finite vs affine W-algebras*, Definition 2.8): Zhu's algebra of the modules twisted by $e^{2\pi i L_0}$, on whose top levels every field has a zero mode. For superconformal algebras whose odd fields have half-integer weights these are the Ramond sectors. The formulas are Zhu's, for elements of any rational weight:

$$A_H(V) = V/J_H, \qquad a * b = \sum_{j\ge0}\binom{\Delta_a}{j} a_{(j-1)}b, \qquad J_H = \operatorname{span}\Big\{\sum_{j\ge0}\binom{\Delta_a}{j} a_{(j-2)}b\Big\}.$$

For a universal vertex algebra it is generated by the images of **all** strong generators, with the relations $[a][b] - (-1)^{|a||b|}[b][a] = \sum_j \binom{\Delta_a - 1}{j}[a_{(j)}b]$, and it is $A_H(V)$ itself when all generators have positive weight (De Sole–Kac, Theorem 3.25), so `is_known_exact()` is `True`. N=1: the Ramond relation $[G]^2 = [L] - c/24$, i.e. $G_0^2 = L_0 - c/24$ on Ramond ground states:

In [31]:
A1h = N1.zhu_algebra(twist='ramond')
print(A1h)
Lh, Gh = A1h.gens()
Gh * Gh, A1h.is_known_exact(), A1h is N1.zhu_algebra(twist='H')

H-twisted Zhu algebra of Universal vertex algebra on generators ['Ln', 'G'] over Fraction Field of Univariate Polynomial Ring in c over Rational Field


(-1/24*c + [Ln], True, True)

N=2: $[J]$ acts on $[G^\pm]$ by $\pm1$, $[G^\pm]^2 = 0$, and $\{[G^+], [G^-]\} = [L] - c/24$:

In [32]:
A2h = N2.zhu_algebra(twist='H')
L2h, J2h, Gph, Gmh = A2h.gens()
print(A2h.relations())
Gph * Gmh + Gmh * Gph == L2h - c/24

{('Gp', 'J2'): -[Gp], ('Gm', 'J2'): [Gm], ('Gm', 'Gp'): -1/24*c + [L2]}


True

Free fields: $\beta\gamma$ with weights $(\frac12, \frac12)$ gives the Weyl algebra (untwisted: $\mathbb C$), $bc$ with weights $(\frac12, \frac12)$ the Clifford algebra. Any rational weights are allowed with the twist, e.g. $(\frac13, \frac23)$, which the untwisted Zhu algebra refuses:

In [33]:
ABGh = BG.zhu_algebra(twist='H')
beh, gah = ABGh.gens()
X.<bf, cf> = NormallyOrderedExpressions(QQ, parities={'bf': 1, 'cf': 1})
BCh = UniversalVertexAlgebra(QQ, X, {('bf', 'cf'): [1]}, degrees={'bf': 1/2, 'cf': 1/2})
bh, ch = BCh.zhu_algebra(twist='H').gens()
BG3 = UniversalVertexAlgebra(QQ, ['be', 'ga'], {('be', 'ga'): [1]}, degrees={'be': 1/3, 'ga': 2/3})
b3, g3 = BG3.zhu_algebra(twist='H').gens()
print(beh * gah - gah * beh, bh * ch + ch * bh, bh * bh, b3 * g3 - g3 * b3)
try:
    BG3.zhu_algebra()
except ValueError as err:
    print('ValueError:', err)

1 1 0 1
ValueError: the conformal weight 1/3 of be is not an integer or half-integer (twist='H' takes any rational weights)


The tricritical Ising model of section 6, now in the Ramond sector: the H-twisted Zhu algebra of the quotient has dimension 4, $[L]$ takes the Ramond weights $\frac{3}{80}$ and $\frac{7}{16}$, and $[G]^2 = [L] - \frac{7}{240}$ is invertible on both, so each Ramond module has a top level of dimension $1|1$:

In [34]:
Z7h = Q7.zhu_algebra_up_to(6, twist='ramond')
Lz, Gz = Z7h.gens()
print(Z7h.dimension(), (Lz - 3/80) * (Lz - 7/16), Lz == 3/80, Lz == 7/16)
Gz * Gz == Lz - 7/240

4 0 False False


True

With integer weights $e^{2\pi i L_0}$ is trivial, and so is the twist: the same algebra. The H-twisted Zhu algebras of W-algebras (finite W-algebras) are in `08_w_algebras.ipynb`, section 8.

In [35]:
Vir.zhu_algebra(twist='H') is Vir.zhu_algebra(), Aff.zhu_algebra(twist='ramond') is Aff.zhu_algebra()

(True, True)

## 11. Higher Zhu algebras

Dong, Li and Mason [DLM1998] define for every $n \ge 0$ an associative algebra $A_n(V)$, with $A_0(V)$ Zhu's algebra: the degrees $0, \dots, n$ of a positive-energy module form an $A_n(V)$-module, and an $A_n(V)$-module generates a module in degree $n$. The library computes $A_n(V)$ for universal vertex algebras with generators of positive integer weight, in He's model [He2017] (a quotient of the degree-zero part of the mode algebra), with `V.zhu_algebra(level=n)`. Its elements print as words in the modes.

For the universal Virasoro algebra at level 1, with $x = [L]$ and $y = L_{-1}L_1$, the algebra is $\mathbb C[x, y]/(y(y - 2x + 2))$:

In [36]:
Kc.<c> = Frac(QQ['c'])
Vc = va.Virasoro(Kc, c, conformal_vector='L'); Lc = Vc.gen('L')
A1 = Vc.zhu_algebra(level=1)
x = A1(Lc); y = (A1(Lc.no(Lc)) - x*x - 2*x)/2
y, y*(y - 2*x + 2), x*y == y*x

(L_{-1} L_1, 0, True)

(Products in $A_n(V)$ are the products of an associative algebra, so `*` is used there as usual.) Quotients up to a weight, modules over $A_n(V)$ and the modules they induce are in the same module (`higher_zhu_algebra.py`); `15_triplet_modules.ipynb` uses the level-1 Zhu algebra of the triplet algebra $\mathcal W(2)$ to find the projective cover $\mathcal P(1)$.

## 12. `TestSuite`

Zhu algebras and their quotients are Sage algebras with basis (super algebras when there are odd generators). `TestSuite` also runs `_test_overlaps`. As for vertex algebras, `TestSuite`'s pickling test needs the classes imported from modules by name, as here.

In [37]:
for Z in [A, AA, AW, ASF, A1, A2, QI.zhu_algebra_up_to(8), ZL1, Zbc, Z0, ZT, ZQ, ZB2,
          A1h, A2h, ABGh, Z7h]:
    TestSuite(Z).run()
print('ok')

ok


## References

- [Zhu1996] Y. Zhu, *Modular invariance of characters of vertex operator algebras*, J. Amer. Math. Soc. 9 (1996), 237–302, [doi:10.1090/S0894-0347-96-00182-8](https://doi.org/10.1090/S0894-0347-96-00182-8). — Zhu's algebra.
- [KW1994] V. G. Kac, W. Wang, *Vertex operator superalgebras and their representations*, Contemp. Math. 175 (1994), 161–191, [arXiv:hep-th/9312065](https://arxiv.org/abs/hep-th/9312065). — half-integer weights.
- [DSK2006] A. De Sole, V. G. Kac, *Finite vs affine W-algebras*, Jpn. J. Math. 1 (2006), 137–261, [arXiv:math-ph/0511055](https://arxiv.org/abs/math-ph/0511055). — the Zhu map, PBW bases, the H-twisted Zhu algebra (sections 1, 10).
- [FZ1992] I. B. Frenkel, Y. Zhu, *Vertex operator algebras associated to representations of affine and Virasoro algebras*, Duke Math. J. 66 (1992), 123–168, [doi:10.1215/S0012-7094-92-06604-X](https://doi.org/10.1215/S0012-7094-92-06604-X). — $A(V/I) = A(V)/[I]$, affine and Virasoro, tensor products.
- [Ber1978] G. M. Bergman, *The diamond lemma for ring theory*, Adv. Math. 29 (1978), 178–218, [doi:10.1016/0001-8708(78)90010-5](https://doi.org/10.1016/0001-8708(78)90010-5). — the diamond lemma (section 3).
- [Wan1993] W. Wang, *Rationality of Virasoro vertex operator algebras*, Int. Math. Res. Not. 1993, no. 7, 197–211, [doi:10.1155/S1073792893000212](https://doi.org/10.1155/S1073792893000212). — Zhu algebras of the Virasoro minimal models (section 6).
- [Don1993] C. Dong, *Vertex algebras associated with even lattices*, J. Algebra 161 (1993), 245–265, [doi:10.1006/jabr.1993.1217](https://doi.org/10.1006/jabr.1993.1217).
- [DLM1997] C. Dong, H. Li, G. Mason, *Regularity of rational vertex operator algebras*, Adv. Math. 132 (1997), 148–166, [arXiv:q-alg/9508018](https://arxiv.org/abs/q-alg/9508018). — lattice vertex algebras (section 8).
- [DLM1998] C. Dong, H. Li, G. Mason, *Vertex operator algebras and associative algebras*, J. Algebra 206 (1998), 67–96, [arXiv:q-alg/9612010](https://arxiv.org/abs/q-alg/9612010) — higher Zhu algebras (section 11).
- [He2017] X. He, *Higher level Zhu algebras are subquotients of universal enveloping algebras*, J. Algebra 491 (2017), 265–279, [arXiv:1712.05671](https://arxiv.org/abs/1712.05671) — the model used for them (section 11).